# docextract: export organised outputs to Drive
Saves Markdown/JSON per document, annotated images, per-page PDF renders, and every table as CSV, in separate folders. Run after `colab_eval.ipynb`, or on its own. Runtime: GPU only needed for the fine-tuned model; CPU is fine for RapidOCR.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/document-capture-runs'
!rm -rf /content/document-capture && git clone -q https://github.com/aamna2401/document-capture.git /content/document-capture
%cd /content/document-capture
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -r requirements-colab.txt

## Data (skip if you ran colab_eval.ipynb in this session)

In [ ]:
import os
os.makedirs('data/public/arxiv', exist_ok=True)
if not os.path.exists('data/public/funsd_raw/dataset'):
    !curl -sL -o data/public/funsd.zip https://guillaumejaume.github.io/FUNSD/dataset.zip
    !unzip -q -o data/public/funsd.zip -d data/public/funsd_raw
for pid in "1706.03762 1810.04805 1512.03385 1409.1556 1412.6980 1502.03167 2010.11929 1706.02677".split():
    if not os.path.exists(f'data/public/arxiv/{pid}.pdf'):
        !curl -sL -o data/public/arxiv/{pid}.pdf https://arxiv.org/pdf/{pid}
        !sleep 3
from datasets import load_dataset
if not os.path.exists('data/cord_test.parquet'):
    load_dataset('naver-clova-ix/cord-v2', split='test').to_parquet('data/cord_test.parquet')
!python scripts/make_gold.py

## Export with the baseline engine (born-digital PDFs have tables; FUNSD/CORD are images; arXiv gold are scanned PDFs)
Start with `--limit 5` to check it works, then raise or remove it.

In [ ]:
OUT = f'{DRIVE}/exports/rapidocr'
!python scripts/export_outputs.py --out {OUT} --engine rapidocr --limit 5 --render-pdf-pages 2 \
    --inputs data/public/arxiv data/gold/arxiv data/gold/funsd data/gold/cord

## Export with the fine-tuned model (after training in colab_eval.ipynb)

In [ ]:
CKPT = f'{DRIVE}/recognizer/best'
if os.path.exists(CKPT):
    !python scripts/export_outputs.py --out {DRIVE}/exports/trocr_finetuned --engine trocr --checkpoint {CKPT} --limit 5 \
        --inputs data/gold/funsd data/gold/cord data/gold/arxiv
else:
    print('no fine-tuned checkpoint yet; train first')

## Check what was saved

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
display(pd.read_csv(f'{OUT}/manifest.csv'))
idx = f'{OUT}/tables/index.csv'
if os.path.exists(idx):
    display(pd.read_csv(idx).head(10))
    display(pd.read_csv(sorted(glob.glob(f'{OUT}/tables/*__page_*.csv'))[0], header=None).head(10))
imgs = sorted(glob.glob(f'{OUT}/images/*/annotated.png'))
if imgs: display(Image(imgs[0], width=500))
!find {OUT} -maxdepth 2 | head -40